# Reproducing Kondo, Sehara, Harukuni, Aoki et al. (2025) from dandiset 001425

Paper: Kondo M, Sehara K, Harukuni R, Aoki R, Sugimoto S, Tanaka YR, Matsuzaki M, Nakae K.
"A multimodal dataset linking wide-field calcium imaging to behavior changes in mice during an
operant lever-pull task", bioRxiv, doi:10.1101/2025.02.03.631599. The claims file
(`claims/001425/10.1101-2025.02.03.631599.yaml`) was written from the preprint, version 1 (JATS
full text with the Methods and the legends of Figures 1 to 6 and S1 to S4; the tables' contents
and the figures themselves were not available to the reader).

Dataset: DANDI:001425, "BraiDyn-BC: mouse cued-lever-pull task dataset", pinned at version
0.250705.0947 (doi:10.48324/dandi.001425/0.250705.0947). The paper cites no version (it
predates the dandiset). There are two published versions, 0.250514.0602 and 0.250705.0947; the
later one is the most recent and the only one that holds the day-0 resting-state sessions of all
25 mice, which claim figs2b-no-aberrant-transients uses (the earlier version lacks those of six
mice; the difference is shown below). So the most recent published version is pinned, as the
protocol asks when the paper cites none. Every file is streamed from the archive; nothing is
downloaded to disk. Loading and the per-session measures are in `bdbc.py`.

Claims: 22 claims. Tested: 20. Blocked before analysis: 2 (data missing). The outcome of each
is in the summary at the end and in the attempt record.

| Claim | Tested or blocked |
|---|---|
| cohort-sex-age | tested |
| task-sessions-collected | tested |
| imaging-frame-counts | tested |
| training-breaks | tested |
| fig3c-response-rate-increase | tested |
| fig3c-success-rate-constant | tested |
| fig3d-tpull-clusters | tested |
| fig3e-cluster-a-high-tpull | tested |
| fig3e-cluster-b-similar-rates | tested |
| methods-no-signal-degradation | tested |
| fig4ab-landmark-validation | blocked: the manual landmark annotations are not in the dataset |
| fig4d-visual-map | tested |
| fig4d-tactile-map | tested |
| fig4d-auditory-map | tested |
| figs2b-no-aberrant-transients | tested |
| fig5abc-dlc-error | blocked: the manual keypoint annotations are not in the dataset |
| fig5f-lever-tip-correlation | tested |
| fig5f-forepaw-correlation | tested |
| fig6b-movement-order | tested |
| fig6b-motor-somatosensory-after-reward | tested |
| fig6b-sensory-areas-track-pupil | tested |
| fig6b-variability-decrease | tested |

The notebook streams about 400 NWB files (only the arrays it needs); with 8 worker processes
this takes about 20 minutes.

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.signal import find_peaks
from scipy.stats import mannwhitneyu, spearmanr, wilcoxon
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

import bdbc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
WORKERS = 8
print("dandiset", bdbc.DANDISET_ID, "version", bdbc.DANDISET_VERSION)
meta = bdbc.version_metadata()
print(meta["name"])
print("described by:", [r.get("url") for r in meta.get("relatedResource", []) if r.get("relation") == "dcite:IsDescribedBy"])
assets = bdbc.list_assets()
print(assets.groupby("kind").size().to_dict())
# the other published version, to show what differs
older = bdbc.list_assets("0.250514.0602")
added = sorted(set(assets.path) - set(older.path)); removed = sorted(set(older.path) - set(assets.path))
both = assets.merge(older, on="path", suffixes=("", "_old"))
print("0.250514.0602 -> 0.250705.0947: added", len(added), "removed", len(removed),
      "same path but different size", int((both["size"] != both["size_old"]).sum()))
print(pd.Series([p for p in added if p.endswith(".nwb")]).str.extract(r"sub-([^/]+)/.*ses-[\d-]+-(.*)_widefield")
      .value_counts().to_string())

## Loading

Four passes over the NWB files, each reduced as it loads (see `bdbc.py`): the metadata and
array shapes of every NWB file; the task sessions (trial table, the 5 kHz lever, tone and reward
channels, the 30 Hz dF/F, keypoints and pupil, 8 raw blue frames, and the 100 Hz body keypoints);
the day-0 resting-state sessions; and the sensory-mapping sessions.

In [ ]:
nwb = assets[assets.kind == "nwb"].reset_index(drop=True)
t0 = time.time()
M = pd.DataFrame(bdbc.pmap(bdbc.session_meta, list(nwb.asset_id), WORKERS))
S = nwb.merge(M, on="asset_id")
S["start"] = pd.to_datetime(S.session_start_time.str[:19])
print("metadata of", len(S), "NWB files read in %.0f s" % (time.time() - t0))

task = S[S.type == "task"].sort_values(["subject", "day"]).reset_index(drop=True)
t0 = time.time()
TASK = bdbc.pmap(bdbc.task_session, list(task.asset_id), WORKERS)
print(len(TASK), "task sessions read in %.0f s" % (time.time() - t0))

rest0 = S[(S.type == "resting-state") & (S.day == 0)].sort_values("subject").reset_index(drop=True)
t0 = time.time()
REST = bdbc.pmap(bdbc.resting_signals, list(rest0.asset_id), WORKERS)
print(len(REST), "day-0 resting-state sessions read in %.0f s" % (time.time() - t0))

sens = S[S.type == "sensory-stim"].sort_values(["subject", "day"]).reset_index(drop=True)
t0 = time.time()
SENS = bdbc.pmap(bdbc.sensory_session, list(sens.asset_id), WORKERS)
print(len(SENS), "sensory-mapping sessions read in %.0f s" % (time.time() - t0))
ROI = S.roi_names.iloc[0].split("|")

In [ ]:
# Per task session: trial outcomes, rates, T_pull, and the checks of the 5 kHz channels.
rows = []
for r, t in zip(task.itertuples(), TASK):
    tr = t["trials"]; n = len(tr); oc = tr.trial_outcome.value_counts()
    rows.append(dict(
        subject=r.subject, day=int(r.day), sex=r.subject_sex, date=r.start.normalize(), n_trials=n,
        success=oc.get("success", 0), failure=oc.get("failure", 0), miss=oc.get("miss", 0),
        resp_a=(oc.get("success", 0) + oc.get("failure", 0)) / n,
        resp_b=tr.lever_above_1mm_in_window.mean(), resp_b_strict=tr.lever_cross_up_in_window.mean(),
        succ_label=oc.get("success", 0) / n, succ_reward=len(t["raw_reward_onsets"]) / n,
        n_reward_ttl=len(t["raw_reward_onsets"]), n_tone_ttl=len(t["raw_tone_onsets"]),
        n_reward_ds=len(t["ds_reward_onsets"]), n_tone_ds=len(t["ds_tone_onsets"]),
        tpull_first=tr.pull_duration_for_success.iloc[0], tpull_final=tr.pull_duration_for_success.iloc[-1],
        lever_min=t["lever_range"][0], lever_max=t["lever_range"][1],
        onset_minus_1mm=np.nanmedian(tr.pull_onset - tr.lever_first_above_t),
        tone_minus_start=np.median([x - s for s in tr.start_time for x in t["raw_tone_onsets"][np.searchsorted(t["raw_tone_onsets"], s - 0.05):][:1]]),
        rt_check=np.nanmax(np.abs(tr.pull_onset - tr.start_time - tr.reaction_time)),
        has_body=t["has_body_kp"], pupil_bad=t.get("pupil_bad_frac", np.nan)))
B = pd.DataFrame(rows)
print(B.shape)

## Data check

What was loaded, and each check of the description against the data.

In [ ]:
print("== files and sessions")
print(S.groupby(["type", "suffix"]).size().to_string())
print("subjects:", S.subject.nunique())
print(pd.crosstab(S.subject, S.type).to_string())
print("\n== subject metadata (one value per subject?)")
g = S.groupby("subject").agg(sex=("subject_sex", lambda x: "|".join(sorted(set(x)))),
                              dob=("subject_date_of_birth", lambda x: "|".join(sorted(set(x)))))
print("sex per subject unique:", (~g.sex.str.contains(r"\|")).all(), "; dob per subject unique:", (~g.dob.str.contains(r"\|")).all())
print(S.subject_genotype.value_counts().to_dict(), S.subject_strain.value_counts().to_dict(), S.subject_species.value_counts().to_dict())
print("session date in file name equals session_start_time date:", int((S.start.dt.normalize() == pd.to_datetime(S.date)).sum()), "of", len(S))
print("\n== imaging")
print("30 Hz dF/F frames by type:"); print(pd.crosstab(S.type, S.n_dff).to_string())
print("blue and violet stacks have the dF/F length in", int(((S.n_blue == S.n_dff) & (S.n_uv == S.n_dff)).sum()), "of", len(S), "files")
print("frame interval (s): %.6f to %.6f" % (S.dff_dt.min(), S.dff_dt.max()))
print("ROIs per file:", S.n_roi.unique(), "; one ROI list across files:", S.roi_names.nunique() == 1)
print("ROIs:", ROI)
print("excitation_lambda of ImagingPlane_blue:", S.blue_lambda.unique(), "; of ImagingPlane_UV:", S.uv_lambda.unique())
print("duplicate recordings (identical first 600 frames of dF/F):", int(S.dff_hash.duplicated().sum()))
print("analysis group contents:", S.analysis.value_counts().to_dict())
print("\n== task channels (364 sessions)")
print("lever unit attribute in the files:", S.lever_unit.dropna().unique())
print("lever range (mm per the code's daq.csv): min %.3f to %.3f, max %.3f to %.3f" % (B.lever_min.min(), B.lever_min.max(), B.lever_max.min(), B.lever_max.max()))
print("T_pull values in trial tables (s):", np.unique(np.round(np.r_[B.tpull_first, B.tpull_final], 3)))
print("trial outcome labels:", sorted(set().union(*[set(t["trials"].trial_outcome) for t in TASK])))
print("trials per session: %d to %d, median %d" % (B.n_trials.min(), B.n_trials.max(), B.n_trials.median()))
print("pull_onset minus first lever sample above 1 mm (s), median per session: %.4f to %.4f" % (B.onset_minus_1mm.min(), B.onset_minus_1mm.max()))
print("max |pull_onset - start_time - reaction_time| (s): %.5f" % B.rt_check.max())
print("raw tone TTL onset minus trial start_time (s), median per session: %.4f to %.4f" % (B.tone_minus_start.min(), B.tone_minus_start.max()))
print("tone TTL onsets minus trials, per session:", (B.n_tone_ttl - B.n_trials).describe()[["min", "50%", "max"]].to_dict())
d = B.n_reward_ttl - B.success
print("reward TTL onsets minus 'success' labels: equal in %d sessions, more TTLs in %d (by 1 to %d, median %d), fewer in %d"
      % ((d == 0).sum(), (d > 0).sum(), d.max(), d[d > 0].median(), (d < 0).sum()))
print("down-sampled reward rising edges minus 'success' labels: median %d, range %d to %d" % ((B.n_reward_ds - B.success).median(), (B.n_reward_ds - B.success).min(), (B.n_reward_ds - B.success).max()))
x = B.n_tone_ds / B.n_trials
print("down-sampled tone rising edges per trial: median %.2f, range %.2f to %.2f; sessions not within 0.9 to 1.1: %d" % (x.median(), x.min(), x.max(), int((~x.between(0.9, 1.1)).sum())))
print("down-sampled reward rising edges differ from 'success' labels in %d sessions" % int((B.n_reward_ds != B.success).sum()))
print("\n== video")
print(pd.crosstab(S.type, S.has_body_kp).rename(columns={True: "body keypoints", False: "none"}).to_string())
kp = S[S.has_body_kp]
print("down-sampled body keypoints: data length equals the 30 Hz length in %d of %d files; their `confidence` has the 30 Hz length in %d"
      % (int((kp.ds_kp_len == kp.n_dff).sum()), len(kp), int((kp.ds_conf_len == kp.n_dff).sum())))
print("task sessions with face keypoints:", int(task.has_face_kp.sum()), "; with pupil:", int(task.has_pupil.sum()))
print("notes:", S.notes.value_counts().to_dict())
pb = B.pupil_bad.dropna()
print("down-sampled pupil diameters not finite, <= 0 or > 300 px: median %.4f of frames, max %.3f; sessions above 5%%: %d of %d"
      % (pb.median(), pb.max(), (pb > 0.05).sum(), len(pb)))

In [ ]:
# Clock checks. Task sessions: the dF/F of left MOp and SSp-ul, 1 s after minus 1 s before the
# lever-pull onsets of successful trials, at the true onsets and at onsets shifted by +/-3 and
# +/-6 s. Sensory sessions: the same contrast at the stimulus onsets of each trial code.
shifts = bdbc.CLOCK_SHIFTS_S
for roi in ["MOp_l", "SSp_ul_l"]:
    k = ROI.index(roi)
    X = np.array([[t["clock"][s][k] for s in shifts] for t in TASK])
    X = X[np.isfinite(X).all(1)]
    print(roi, "median over", len(X), "task sessions with successful pulls:", dict(zip(shifts, np.round(np.nanmedian(X, 0), 4))),
          "; sessions where the true onsets give the largest contrast:", int((np.nanargmax(X, 1) == shifts.index(0.0)).sum()))
for key, roi in [("code:1", "VISp_r"), ("code:3", "SSp_bfd_l"), ("code:2", "AUDs_l")]:
    k = ROI.index(roi)
    X = np.array([[s_["clock"][key][s][k] for s in shifts] for s_ in SENS if key in s_.get("clock", {})])
    X = X[np.isfinite(X).all(1)]
    print(key, roi, "median over", len(X), "sensory sessions:", dict(zip(shifts, np.round(np.nanmedian(X, 0), 4))),
          "; sessions where the true onsets give the largest contrast:", int((np.nanargmax(X, 1) == shifts.index(0.0)).sum()))

Checks of the description against the data, as the outputs above show them:

- The files hold 364 task, 98 resting-state and 33 sensory-mapping sessions of 25 subjects; each
  subject has one sex and one date of birth; the session date in each file name matches its
  `session_start_time`. No two files share their first 600 frames of dF/F, so no recording was
  found deposited twice.
- The 30 Hz series have the frame interval of 30 Hz, and the raw blue and violet stacks have the
  length of the dF/F in every file. Each file has the same 44 ROIs (22 per hemisphere).
- In every file `ImagingPlane_blue` states an excitation wavelength of 405 nm and
  `ImagingPlane_UV` 470 nm, while their own descriptions (and the Methods) say the reverse; the
  data are taken by their names and descriptions (`widefield_blue`, `dFF_B` "473 nm excitation").
- The lever is stored with unit "a.u."; its range (about -0.4 to 4.3) fits the 4 mm travel of the
  Methods and the unit mm that the paper's code gives (bdbc-nwb-explorer, data/daq.csv).
- The trial table's `pull_onset` follows the first lever sample above 1 mm by about 14 ms in every
  session (the online threshold), and `reaction_time` is `pull_onset - start_time`. The raw tone
  TTL rises 2 to 18 ms after `start_time` (median per session). Most sessions have a tone TTL or
  two more than trials, and a few have many more (the counts printed).
- Reward TTLs outnumber the "success" labels in more than half of the sessions, usually by one,
  in a few sessions by tens, so the two ways of counting successes differ; both are shown. The
  down-sampled `tone` series rises once per trial in every session, but the down-sampled `reward`
  series differs from the success labels in 44 sessions (the counts printed), so events are taken
  from the 5 kHz channels and the trial table.
- The down-sampled keypoint series keep a `confidence` array of the 100 Hz length next to 30 Hz
  data (counts printed), so the two cannot be paired.
- Some pupil diameters are not finite or far wider than the eye video (values of 1e5 pixels and
  more); these are set missing (`bdbc.PUPIL_MAX_PX`).
- Clock: in the task sessions the motor-area response is largest at the true lever-pull onsets,
  and in the sensory sessions the visual and tactile responses are largest at the true stimulus
  onsets, in nearly every session (the numbers printed above); the code-2 response in left AUDs is
  small and peaks at the true onsets in only 15 of 27 sessions, so for that stimulus the check is
  weak; the 100 Hz body keypoints and the 5 kHz lever share a clock
  as far as the lever-tip correlation shows (claim fig5f-lever-tip-correlation). The resting-state
  sessions have no events, so their clock could not be checked this way; their analysis does not
  use events.

## cohort-sex-age

> The dataset comes from 25 mice of both sexes, 11 males and 14 females, aged 17 to 31 weeks at
> the end of the experiments.

Criterion: 25 subjects, 11 male and 14 female, every subject's age at its last session within
15.3 to 34.1 weeks, from the subject metadata (age or date of birth) and the session dates.

In [ ]:
last = S.sort_values("start").groupby("subject").tail(1).set_index("subject")
dob = pd.to_datetime(last.subject_date_of_birth.str[:10])
age_dob_w = (last.start.dt.normalize() - dob).dt.days / 7
age_field_w = last.subject_age.str.extract(r"P(\d+)D")[0].astype(int) / 7
sexes = S.groupby("subject").subject_sex.first()
print("subjects:", sexes.size, "; male:", int((sexes == "M").sum()), "; female:", int((sexes == "F").sum()))
tab = pd.DataFrame({"last session": last.type + " day " + last.day.astype(int).astype(str),
                    "age from date of birth (weeks)": age_dob_w.round(1), "age field (weeks)": age_field_w.round(1)})
print(tab.to_string())
for nm, a in [("date of birth", age_dob_w), ("age field", age_field_w)]:
    print("%s: %.1f to %.1f weeks; all within 15.3 to 34.1: %s" % (nm, a.min(), a.max(), bool(((a >= 15.3) & (a <= 34.1)).all())))

**Result: reproduced.** The files hold 25 subjects, 11 male and 14 female, and every subject's
age at its last session, from the date of birth and from the age field, lies inside the widened
range of 15.3 to 34.1 weeks (the printed range).

## task-sessions-collected

> Of the 375 task training sessions (25 mice by 15 sessions), data were collected from 364 (97%);
> the 11 others lacked complete imaging. All 15 sessions were collected for 16 of the 25 mice.

Criterion: task sessions identified by their description or file name; recordings deposited more
than once count once (none were found, above); 364 distinct task sessions and 16 mice with all 15.

In [ ]:
is_task_desc = S.session_description.str.startswith("Lever-pull task")
print("task by file name:", int((S.type == "task").sum()), "; by session description:", int(is_task_desc.sum()),
      "; both agree:", bool(((S.type == "task") == is_task_desc).all()))
per = task.groupby("subject").day.apply(lambda d: sorted(int(x) for x in d))
missing = per.apply(lambda d: [x for x in range(1, 16) if x not in d])
print("distinct task sessions:", task.drop_duplicates(["subject", "day"]).shape[0])
print("mice with all 15 task sessions:", int((per.apply(len) == 15).sum()))
print("missing task-session numbers per mouse:"); print(missing[missing.apply(len) > 0].to_string())
print("missing in total:", int(missing.apply(len).sum()))
rs = S[S.type == "resting-state"].groupby("subject").day.apply(lambda d: sorted(int(x) for x in d))
print("resting-state sessions:", int((S.type == "resting-state").sum()), "; per mouse:", rs.apply(len).value_counts().to_dict())
print("mice with fewer than 4 resting-state sessions:", rs[rs.apply(len) < 4].to_dict())
print("sensory-mapping sessions:", len(sens), "; mice with two:", int((sens.groupby("subject").size() == 2).sum()))

**Result: reproduced.** There are 364 distinct task sessions and 16 mice with all 15; the 11
missing sessions are listed. The dataset also holds 98 resting-state sessions (two mice have three,
missing the day-15 session) and 33 sensory-mapping sessions (8 mice have two), the counts printed.

## imaging-frame-counts

> Each task session was imaged for 30 min (108000 frames at 60 Hz, so 54000 frames per excitation
> channel at 30 Hz), each resting-state session for 10 min and the sensory-mapping session for 15
> min.

Criterion: most sessions of each type have 54000, 18000 and 27000 frames per channel; sessions
that differ are listed.

In [ ]:
expect = {"task": 54000, "resting-state": 18000, "sensory-stim": 27000}
for ty, n in expect.items():
    s = S[S.type == ty]
    print("%s: %d of %d sessions have %d frames (dF/F and both stacks)" % (ty, int(((s.n_dff == n) & (s.n_blue == n) & (s.n_uv == n)).sum()), len(s), n))
dev = S[S.n_dff != S.type.map(expect)][["subject", "type", "day", "n_dff", "notes"]]
print(dev.to_string())

**Result: reproduced.** 363 of 364 task, all 98 resting-state and 29 of 33 sensory-mapping sessions
have the stated lengths; the five that differ are listed (one task session of 45413 frames, noted in
its file as 25.2 min, and four sensory-mapping sessions of 36000 frames, 20 min).

## training-breaks

> Training had 3.1 +/- 0.6 breaks per mouse of 2.3 +/- 0.7 days each (mean +/- s.d., 25 mice), and
> recording days 1 to 16 spanned 24.28 +/- 3.2 calendar days.

Criterion: dates are session start dates; a break is a run of calendar days without a task session
between consecutive task sessions (days 1 to 15). Readings: (1) breaks of (a) one or more days,
(b) two or more days; (2) days per break pooled over breaks or the mean of per-mouse means; (3)
day 16 as the first or the last sensory-mapping session; the span inclusive or as a difference.
Tolerances are the paper's s.d. Gaps that enclose a missing (not deposited) task-session number
are listed, and a variant leaves them out.

In [ ]:
br = []
for subj, g in B.sort_values("day").groupby("subject"):
    d = g.date.to_numpy(); day = g.day.to_numpy()
    for i in range(1, len(d)):
        gap = int((d[i] - d[i - 1]) / np.timedelta64(1, "D")) - 1
        if gap > 0:
            br.append(dict(subject=subj, after_day=day[i - 1], before_day=day[i], days=gap, encloses_missing=day[i] - day[i - 1] > 1))
BR = pd.DataFrame(br)
print("gaps that enclose a missing task-session number:"); print(BR[BR.encloses_missing].to_string())
mice = sorted(B.subject.unique())
res = []
for minlen in [1, 2]:
    for excl in [False, True]:
        b = BR[(BR.days >= minlen) & (~BR.encloses_missing if excl else True)]
        nper = b.groupby("subject").size().reindex(mice, fill_value=0)
        pooled = b.days.mean(); permouse = b.groupby("subject").days.mean().mean()
        res.append(dict(reading=f"(1{'a' if minlen == 1 else 'b'}) >= {minlen} day", leave_out_missing=excl,
                        breaks_mean=nper.mean(), breaks_sd=nper.std(), days_pooled=pooled, days_pooled_sd=b.days.std(),
                        days_permouse=permouse))
R = pd.DataFrame(res).round(2)
R["breaks in 3.1+/-0.6"] = R.breaks_mean.between(2.5, 3.7)
R["days in 2.3+/-0.7 (pooled)"] = R.days_pooled.between(1.6, 3.0)
R["days in 2.3+/-0.7 (per mouse)"] = R.days_permouse.between(1.6, 3.0)
print(R.to_string())
print("break lengths (days), reading (1a):", BR.days.value_counts().sort_index().to_dict())
d1 = B[B.day == 1].set_index("subject").date
s16 = sens.groupby("subject").start
spans = pd.DataFrame({"first": (s16.min().dt.normalize() - d1).dt.days, "last": (s16.max().dt.normalize() - d1).dt.days}).dropna()
print("mice with a day-1 task session and a sensory-mapping session:", len(spans))
for c in ["first", "last"]:
    for incl in [0, 1]:
        x = spans[c] + incl
        print("span, day 16 = %s sensory session, %s: mean %.2f, s.d. %.2f; within 24.28+/-3.2: %s"
              % (c, "inclusive" if incl else "difference", x.mean(), x.std(), bool(abs(x.mean() - 24.28) <= 3.2)))

**Result: reproduced.** Under every reading in the table (breaks of one or more days or of two or
more, with or without the gaps that enclose a missing task session) the mean number of breaks per
mouse and the mean days per break, pooled or per mouse, are within the paper's s.d. The span from
the day-1 task session to the sensory-mapping session (24 mice; one has no day-1 session) is within
24.28 +/- 3.2 days under every reading printed. One break lasts 16 days, which makes the s.d. of
break lengths larger than the paper's 0.7.

## fig3c-response-rate-increase

> Across the population the response rate to the tone rose over training, from about 0.55 in task
> session 1 to about 0.85 in task session 15 (median over mice).

Criterion: response rate = valid pulls / tones, per session, median over the mice with that
session. Readings: (a) trial-table labels, success and failure valid; (b) the lever above 1 mm
within 1 s of tone onset (the 5 kHz lever; a strict upward crossing is shown as a variant).
Tolerances 0.055 and 0.085.

In [ ]:
med = B.groupby("day")[["resp_a", "resp_b", "resp_b_strict", "succ_label", "succ_reward", "tpull_final"]].median()
med["n_mice"] = B.groupby("day").size()
print(med.round(3).to_string())
for c in ["resp_a", "resp_b", "resp_b_strict"]:
    s1, s15 = med.loc[1, c], med.loc[15, c]
    print("%s: session 1 %.3f (0.55+/-0.055: %s), session 15 %.3f (0.85+/-0.085: %s), rise: %s"
          % (c, s1, abs(s1 - 0.55) <= 0.055, s15, abs(s15 - 0.85) <= 0.085, s15 > s1))
print("by sex, reading (a):"); print(B.groupby(["day", "sex"]).resp_a.median().unstack().loc[[1, 15]].round(3).to_string())
print("rho of the 15 medians with session, reading (a): %.3f (p = %.4f)" % spearmanr(med.index, med.resp_a))
fig, ax = plt.subplots(figsize=(5, 3))
for s, g in B.groupby("subject"):
    ax.plot(g.day, g.resp_a, color="0.8", lw=0.7)
ax.plot(med.index, med.resp_a, "k", lw=2); ax.set_xlabel("task session"); ax.set_ylabel("response rate (a)"); ax.set_ylim(0, 1.02)
print("plotted range of per-mouse response rates: %.3f to %.3f" % (B.resp_a.min(), B.resp_a.max()))

**Result: reproduced.** Under both readings the median response rate is within tolerance of 0.55
in session 1 and of 0.85 in session 15, and higher in session 15 (the printed values).

## fig3c-success-rate-constant

> The population success rate stayed roughly constant across the 15 task sessions.

Criterion: success rate = successful pulls / tones, median over mice. Readings: (a) session-15
median within 10% of the session-1 median; (b) no monotonic trend of the 15 medians, Spearman
p >= 0.05. Successes from reward TTLs and from the outcome labels. Holds when both readings hold,
fails when both fail, blocked as methods_underspecified when they differ.

In [ ]:
for c in ["succ_label", "succ_reward"]:
    s1, s15 = med.loc[1, c], med.loc[15, c]
    rho, p = spearmanr(med.index, med[c])
    print("%s: session 1 %.3f, session 15 %.3f, change %+.1f%% -> (a) within 10%%: %s; (b) Spearman rho %.3f, p %.3f -> no trend: %s"
          % (c, s1, s15, 100 * (s15 / s1 - 1), abs(s15 / s1 - 1) <= 0.10, rho, p, p >= 0.05))
print("range of the 15 medians, labels: %.3f to %.3f; rewards: %.3f to %.3f" % (med.succ_label.min(), med.succ_label.max(), med.succ_reward.min(), med.succ_reward.max()))
print("for contrast, response rate (a) change session 1 to 15: %+.1f%%" % (100 * (med.loc[15, 'resp_a'] / med.loc[1, 'resp_a'] - 1)))
rho_m = B.groupby("subject").apply(lambda g: spearmanr(g.day, g.succ_label).statistic)
print("per mouse, Spearman rho of success rate (labels) with session: median %.3f, positive in %d of %d" % (rho_m.median(), (rho_m > 0).sum(), len(rho_m)))

**Result: blocked (methods_underspecified).** The two readings of "constant" disagree: the 15
session medians show no monotonic trend (reading (b), p >= 0.05), but the session-15 median is more
than 10% above the session-1 median (reading (a)), with successes counted either way. The
criterion blocks the claim when the readings differ.

## fig3d-tpull-clusters

> Hierarchical clustering (Ward's method) of the session-by-session T_pull_final divides the mice
> into two clusters, A with 7 mice and B with 18.

Criterion: T_pull_final from the trial table's required-duration field at the last trial; Ward's
method on Euclidean distance, cut at two clusters; A has the higher mean T_pull_final. Readings:
(1) the 15 values or the 14 differences; (2) missing sessions by (i) linear interpolation, (ii)
previous value carried forward, (iii) the next session's initial T_pull plus 100 ms.

In [ ]:
P = B.pivot(index="subject", columns="day", values="tpull_final")
F = B.pivot(index="subject", columns="day", values="tpull_first")
def impute(how):
    X = P.copy()
    for s in X.index:
        row = P.loc[s]
        if how == "interp":
            X.loc[s] = row.interpolate(limit_direction="both")
        elif how == "ffill":
            X.loc[s] = row.ffill().bfill()
        else:
            fill = row.ffill().bfill()
            for d in row.index[row.isna()]:
                X.loc[s, d] = F.loc[s, d + 1] + 0.1 if (d + 1 in F.columns and np.isfinite(F.loc[s, d + 1])) else fill[d]
    return X
CLUSTERS = {}
out = []
for how in ["interp", "ffill", "next"]:
    X = impute(how)
    for feat in ["values", "differences"]:
        Z = X.to_numpy() if feat == "values" else np.diff(X.to_numpy(), axis=1)
        lab = fcluster(linkage(Z, "ward"), 2, "maxclust")
        a = 1 if X.to_numpy()[lab == 1].mean() > X.to_numpy()[lab == 2].mean() else 2
        A_ = sorted(X.index[lab == a])
        CLUSTERS[(how, feat)] = A_
        out.append(dict(missing=how, feature=feat, n_A=len(A_), n_B=len(X) - len(A_), match=(len(A_) == 7), A=", ".join(A_)))
print(pd.DataFrame(out).to_string())
print("\nT_pull_final (s) per mouse and session (NaN = not deposited):")
print(P.round(2).to_string())

**Result: reproduced.** Clustering the 15 T_pull_final values gives clusters of 7 and 18 mice under
all three ways of filling the missing sessions, with the same 7 mice in cluster A. Clustering the
14 differences does not give 7 and 18 (the table).

## fig3e-cluster-a-high-tpull

> Mice of cluster A had high success rates in the early sessions (about session 5), which led to
> a high T_pull_final that was often maintained into the late sessions.

Criterion: clusters of fig3d-tpull-clusters (all readings giving 7 and 18 have the same members);
(1) A's median success rate above B's in (a) sessions 1 to 5 (per-mouse mean), (b) session 5;
(2) A's median T_pull_final above B's in (a) sessions 11 to 15, (b) session 15.

In [ ]:
A_MICE = CLUSTERS[("interp", "values")]
assert all(v == A_MICE for k, v in CLUSTERS.items() if len(v) == 7)
B["cluster"] = np.where(B.subject.isin(A_MICE), "A", "B")
def per_mouse(col, days):
    return B[B.day.isin(list(days))].groupby(["cluster", "subject"])[col].mean()
for col in ["succ_label", "succ_reward"]:
    for nm, days in [("sessions 1-5", range(1, 6)), ("session 5", [5])]:
        x = per_mouse(col, days)
        print("early success (%s), %s: A median %.3f (n=%d), B median %.3f (n=%d), A higher: %s, Mann-Whitney p %.3f"
              % (col, nm, x["A"].median(), x["A"].size, x["B"].median(), x["B"].size, x["A"].median() > x["B"].median(), mannwhitneyu(x["A"], x["B"]).pvalue))
for nm, days in [("sessions 11-15", range(11, 16)), ("session 15", [15])]:
    x = per_mouse("tpull_final", days)
    print("late T_pull_final, %s: A median %.3f s, B median %.3f s, A higher: %s" % (nm, x["A"].median(), x["B"].median(), x["A"].median() > x["B"].median()))
print(B.groupby(["cluster", "day"]).tpull_final.median().unstack().round(2).to_string())
print(B.groupby(["cluster", "day"]).succ_label.median().unstack().round(2).to_string())

**Result: reproduced.** Cluster A's median success rate is above cluster B's in sessions 1 to 5
and in session 5, with successes counted either way, and its median T_pull_final is above B's in
sessions 11 to 15 and in session 15 (the printed medians). Part (2) follows largely from naming A as
the cluster with the higher mean T_pull_final, as the claims file notes.

## fig3e-cluster-b-similar-rates

> Mice of cluster B rarely reached a high T_pull_final, but in the late sessions their response and
> success rates were similar to those of cluster A.

Criterion: "high" T_pull_final read as (a) 400 ms, (b) above the median of all mouse-session values;
"rarely" when B's fraction of mouse-sessions above that level is below A's and below one half.
Late sessions (a) 11 to 15 (per-mouse mean), (b) 15. "Similar" read as (a) Mann-Whitney p >= 0.05,
(b) group medians within 10% of A's. Holds when the T_pull_final part holds and, for both rates,
the similarity holds under both readings of "similar"; blocked as methods_underspecified when
those readings differ.

In [ ]:
allmed = B.tpull_final.median()
for nm, lvl in [("(a) 400 ms", 0.4 - 1e-6), ("(b) above median %.2f s" % allmed, allmed)]:
    fa = (B[B.cluster == "A"].tpull_final > lvl).mean(); fb = (B[B.cluster == "B"].tpull_final > lvl).mean()
    print("high %s: fraction of mouse-sessions A %.3f, B %.3f -> rarely: %s" % (nm, fa, fb, (fb < fa) and (fb < 0.5)))
rows = []
for col in ["resp_a", "resp_b", "succ_label", "succ_reward"]:
    for nm, days in [("11-15", range(11, 16)), ("15", [15])]:
        x = per_mouse(col, days); a, b = x["A"], x["B"]
        p = mannwhitneyu(a, b).pvalue; rel = abs(b.median() - a.median()) / a.median()
        rows.append(dict(rate=col, late=nm, A=a.median(), B=b.median(), nA=a.size, nB=b.size, p=p, rel_diff=rel,
                         similar_a=p >= 0.05, similar_b=rel <= 0.10))
SIM = pd.DataFrame(rows)
print(SIM.round(3).to_string())

**Result: reproduced.** Cluster B reached 400 ms in none of its mouse-sessions and was above the
overall median in about a fifth of them, against most of cluster A's, so the T_pull_final part
holds under both readings. In the late sessions the response rates of the two clusters are similar
under both readings of "similar". For the success rate, the Mann-Whitney reading holds everywhere,
and the 10% reading holds with successes counted from reward TTLs over sessions 11 to 15 but not
with the outcome labels or for session 15 alone (cluster B's median is the higher one). So every
part holds under one set of readings, rewards counted and sessions 11 to 15, and the readings of
"similar" differ for the success rate under the others; the table shows each.

## methods-no-signal-degradation

> Fluorescence signal degradation across imaging sessions was not observed.

Criterion: (a) the raw 470 nm fluorescence, the ROI baseline F_B0 (median over time) or the
session-mean intensity of the blue stack within the ROI masks; (b) the s.d. of the corrected dF/F
per ROI; each against task-session index, per-mouse Spearman correlation, Wilcoxon signed-rank test
of the 25 correlations against 0 at p < 0.05. Holds when neither finds a significant decline. Here
F_B0 is the median over 8 evenly spaced blue frames of each ROI's mean intensity, averaged over
the 44 ROIs (`bdbc.N_BLUE_FRAMES`; reading all 54000 frames does not fit the time limit).

In [ ]:
rows = []
for r, t in zip(task.itertuples(), TASK):
    rows.append(dict(subject=r.subject, day=int(r.day), fb0=np.median(t["blue_roi"], 0).mean(), sd=t["dff_sd"].mean(),
                     fb0_roi=np.median(t["blue_roi"], 0), sd_roi=t["dff_sd"]))
DG = pd.DataFrame(rows)
for col, nm in [("fb0", "(a) raw blue F_B0"), ("sd", "(b) s.d. of corrected dF/F")]:
    rho = DG.groupby("subject").apply(lambda g: spearmanr(g.day, g[col]).statistic)
    w = wilcoxon(rho)
    ratio = (DG[DG.day == 15].set_index("subject")[col] / DG[DG.day == 1].set_index("subject")[col]).dropna()
    print("%s: median rho %.3f, negative in %d of %d mice, Wilcoxon p %.3f -> significant decline: %s; session 15 / session 1 median %.3f over %d mice"
          % (nm, rho.median(), (rho < 0).sum(), len(rho), w.pvalue, (rho.median() < 0) and (w.pvalue < 0.05), ratio.median(), len(ratio)))
    print("   per mouse rho:", rho.round(2).to_dict())
for col in ["fb0_roi", "sd_roi"]:
    sig_dec = sig_inc = 0
    for k in range(44):
        rho = DG.groupby("subject").apply(lambda g: spearmanr(g.day, np.stack(g[col])[:, k]).statistic)
        p = wilcoxon(rho).pvalue
        sig_dec += (rho.median() < 0) and p < 0.05; sig_inc += (rho.median() > 0) and p < 0.05
    print("variant per ROI, %s: ROIs with a significant decline %d, increase %d, of 44 (uncorrected)" % (col, sig_dec, sig_inc))
print("F_B0 range over sessions: %.0f to %.0f (camera counts); s.d. range %.4f to %.4f" % (DG.fb0.min(), DG.fb0.max(), DG.sd.min(), DG.sd.max()))

**Result: reproduced.** Neither reading finds a significant decline over the 15 task sessions
(Wilcoxon p >= 0.05 for both); the session-15 over session-1 ratios are printed as the size of the
change. Per ROI, a few of the 44 ROIs show a nominal decline without correction for 44 tests (the
variant).

## fig4ab-landmark-validation (blocked)

> MesoNet-inferred skull landmarks used to register the Allen CCF agreed with manual annotations of
> three representative sessions from each of the 25 mice (75 sessions).

The test needs the manual landmark annotations. The NWB files' `analysis` group holds only the
atlas-to-data affine transform (printed in the data check); the dandiset holds no other files than
the NWB files and videos, and the repositories the paper and the dandiset link
(BraiDyn-BC/bdbc-data-pipeline, bdbc-nwb-packager, bdbc-nwb-explorer, bdbc-session-explorer) hold
no annotation files. Blocked: data missing.

## fig4d-visual-map, fig4d-tactile-map, fig4d-auditory-map

> In the sensory-mapping sessions, visual stimulation of the left eye evoked the strongest activity
> in the right occipital cortex, at VISp in the registered atlas.

> Vibration of the right whisker pad evoked a robust response in the left barrel cortex
> (SSp-bfd), with marked laterality.

> White-noise stimulation evoked spreading activity that included a multimodal sensory area in
> the parietal cortex.

Criterion: first sensory-mapping session of each mouse; trial-averaged response over 2 s from
stimulus onset, normalized 0 to 1 per mouse and stimulus; the area above the 80th percentile.
Readings: (1) the 44 ROI dF/F series, or pixel maps of the blue stack; (2) no baseline, or the 2 s
before onset subtracted. Stimulus onsets from the stimulus TTLs.

The files do not say which `stim_modality` code is which stimulus, and the TTL channels are not
consistent across sessions (below), so the maps are computed both for each TTL channel as named
and for each trial code. The pixel-map reading was not run: the stacks have one 288 x 288 frame
per chunk, read at about 0.6 s per frame, so the 60 to 120 frames around each of about 30 stimuli
per modality take about an hour per session.

In [ ]:
first = sens.groupby("subject").head(1).index
print("first sessions:", len(first), "; second sessions not used:", len(sens) - len(first))
pat = []
for i in first:
    s_ = SENS[i]
    row = {"subject": sens.subject[i], "frames": s_["n_frames"]}
    for ch in bdbc.SENSORY_TTL:
        row[ch + " pulses/stim"] = round(s_["pulses"].get(ch, np.nan), 1)
    tr = s_["trials"]
    for ch in bdbc.SENSORY_TTL:
        on = s_.get("ch:" + ch + "_onsets")
        if on is None or not len(on):
            continue
        k = np.searchsorted(tr.start_time.to_numpy(), on - 0.05)
        k = np.clip(k, 0, len(tr) - 1)
        row[ch + " -> code"] = "/".join(str(c) for c in sorted(set(tr.stim_modality.to_numpy()[k])))
    row["codes"] = "/".join(str(c) for c in sorted(tr.stim_modality.unique()))
    row["order of first three codes"] = "-".join(str(c) for c in tr.stim_modality.to_numpy()[:3])
    pat.append(row)
PAT = pd.DataFrame(pat)
print(PAT.to_string())

In [ ]:
def sensory_maps(key, base):
    maps, tops, subj = [], [], []
    for i in first:
        r = SENS[i]["resp"].get(key)
        if r is None:
            continue
        x = r[base]; xn = (x - x.min()) / (x.max() - x.min())
        maps.append(xn); tops.append(ROI[int(np.argmax(xn))]); subj.append(sens.subject[i])
    m = np.mean(maps, 0)
    region = [ROI[k] for k in np.flatnonzero(m > np.percentile(m, 80))]
    return m, region, pd.Series(tops, index=subj), np.array(maps)
SMAP = {}
for key in ["ch:stim_visual", "ch:stim_whisker", "ch:stim_auditory", "code:1", "code:2", "code:3"]:
    for base in ["none", "baseline"]:
        m, region, tops, maps = sensory_maps(key, base)
        SMAP[(key, base)] = (m, region, tops, maps)
        order = np.argsort(m)[::-1]
        print("%-17s %-8s n=%2d top: %s | 80th-percentile region: %s | per-mouse top ROI: %s"
              % (key, base, len(tops), ", ".join("%s %.2f" % (ROI[k], m[k]) for k in order[:4]), ", ".join(region), tops.value_counts().head(3).to_dict()))

In [ ]:
def judge_visual(key, base):
    m, region, tops, maps = SMAP[(key, base)]
    return ROI[int(np.argmax(m))] == "VISp_r", int((tops == "VISp_r").sum()), len(tops)
for key in ["ch:stim_visual", "code:1"]:
    for base in ["none", "baseline"]:
        ok, n, N = judge_visual(key, base)
        print("visual, %s, %s: right VISp has the largest mean response: %s; in %d of %d mice" % (key, base, ok, n, N))

**Result: reproduced.** The 10 Hz flash train (code 1; the `stim_visual` TTL in the 19 sessions
that have it) evokes its largest mean response in right VISp under both baselines, with right VISpm
close behind (the printed values); per mouse, right VISp is the top ROI in 8 to 11 mice and right
VISpm about as often. The pixel-map reading was not run.

In [ ]:
il, ir = ROI.index("SSp_bfd_l"), ROI.index("SSp_bfd_r")
for key in ["ch:stim_whisker", "code:3"]:
    for base in ["none", "baseline"]:
        m, region, tops, maps = SMAP[(key, base)]
        print("tactile, %s, %s: left SSp-bfd largest: %s; left %.2f vs right %.2f; left > right in %d of %d mice; left SSp-bfd top in %d mice"
              % (key, base, ROI[int(np.argmax(m))] == "SSp_bfd_l", m[il], m[ir], int((maps[:, il] > maps[:, ir]).sum()), len(maps), int((tops == "SSp_bfd_l").sum())))

**Result: reproduced.** The stimulus on the `stim_whisker` channel, and trial code 3, evoke their
largest mean response in left SSp-bfd, well above right SSp-bfd, under both baselines (the printed
values). In the six sessions without a `stim_visual` channel the `stim_whisker` channel carries the
10 Hz visual train, so the channel-name reading mixes six visual maps into the 25; the code-3
reading, which matches the `stim_whisker` channel in the other 19 sessions and the `stim_auditory`
channel in those six, gives the same result.

In [ ]:
PARIETAL = {"VISa_l", "VISa_r", "VISrl_l", "VISrl_r"}
for key in ["ch:stim_auditory", "code:2"]:
    for base in ["none", "baseline"]:
        m, region, tops, maps = SMAP[(key, base)]
        print("auditory, %s, %s: n=%d, region %s; parietal ROI in region: %s; more than one ROI: %s; range of mean map %.2f to %.2f"
              % (key, base, len(maps), region, bool(PARIETAL & set(region)), len(region) > 1, m.min(), m.max()))
print("no ROI named PTLp in the files:", not any("PTLp" in r for r in ROI))

**Result: reproduced.** Under both readings of the stimulus (the `stim_auditory` channel as named,
and code 2, which exists only in the 19 sessions with three stimulus channels) and both baselines,
the 80th-percentile region of the mean map spans several ROIs and includes VISa, a posterior parietal
ROI. The mean map is flat (the printed range), so "spreading" holds, but no area stands out. In the
six sessions without `stim_visual`, the `stim_auditory` channel carries trial code 3, whose map
peaks in left SSp-bfd, so the channel-name reading mixes six tactile maps into the 25.

## figs2b-no-aberrant-transients

> The mice showed no sign of the aberrant short-duration, large-amplitude calcium transients
> reported in some transgenic lines; no cluster of transients had high prominence (> 0.1 dF/F) and
> narrow width (< 0.5 s).

Criterion: each mouse's day-0 resting-state recording; left frontal (MOs or MOp), left barrel
(SSp-bfd), left retrosplenial (RSPd or RSPagl), right visual (VISp); transients detected with
`scipy.signal.find_peaks` (a) with no threshold, (b) minimum prominence 0.01; width at half
prominence at 30 Hz; per mouse and region K-means (k = 2) on (prominence, width), raw or z-scored;
signal corrected dF/F or uncorrected dF_B/F_B. Reproduced when no cluster centre has prominence
> 0.1 and width < 0.5 s.

In [ ]:
res = []
for r, subj in zip(REST, rest0.subject):
    for sig in ["dff", "dff_b"]:
        for j, roi in enumerate(r["rois"]):
            x = r[sig][:, j]
            for thr in [None, 0.01]:
                pk, pr = find_peaks(x, prominence=(thr if thr else 0), width=0, rel_height=0.5)
                X = np.c_[pr["prominences"], pr["widths"] / bdbc.FS]
                for feat in ["raw", "z"]:
                    Xf = X if feat == "raw" else (X - X.mean(0)) / X.std(0)
                    lab = KMeans(2, n_init=10, random_state=0).fit(Xf).labels_
                    cen = np.array([X[lab == k].mean(0) for k in range(2)])
                    res.append(dict(subject=subj, signal=sig, roi=roi, min_prominence=thr or 0, features=feat, n_transients=len(pk),
                                    max_centre_prominence=cen[:, 0].max(), narrow_centre_width=cen[np.argmax(cen[:, 0]), 1],
                                    aberrant=bool(((cen[:, 0] > 0.1) & (cen[:, 1] < 0.5)).any())))
TR = pd.DataFrame(res)
print("mice:", TR.subject.nunique(), "; region x signal x reading combinations:", len(TR))
print(TR.groupby(["signal", "min_prominence", "features"]).agg(aberrant=("aberrant", "sum"), runs=("aberrant", "size"),
      median_transients=("n_transients", "median"), max_centre_prominence=("max_centre_prominence", "max"),
      min_width_of_high_centre=("narrow_centre_width", "min")).round(3).to_string())
print(TR.groupby("subject").n_transients.median().describe().round(0).to_dict())

**Result: reproduced.** In none of the 25 mice, 6 ROIs, 2 signals and 4 detection and feature
readings does a K-means centre have prominence above 0.1 dF/F with width below 0.5 s (the table
prints 0 aberrant runs in each group).

## fig5abc-dlc-error (blocked)

> DeepLabCut keypoints differed from manual annotations only by minor errors on validation frames
> (192 frames from 25 mice for the body and face cameras, 190 for the eye).

The manual annotations of the validation frames are not in the dataset (the files hold only the
DeepLabCut estimates and likelihoods) and are not in the repositories the paper links. Blocked:
data missing.

## fig5f-lever-tip-correlation and fig5f-forepaw-correlation

> The lever-tip position tracked from the body video correlated highly with the encoder lever
> position, R = 0.98 +/- 0.02 (mean +/- s.d.).

> The right forepaw, which pulls the lever, correlated with the lever position (R = 0.49 +/- 0.23),
> and the left forepaw did not (R = 0.05 +/- 0.19), over 354 task sessions from 25 mice.

Criterion: Pearson R between the keypoint's y position and the encoder lever over the frames of all
successful trials, per task session with body video and at least one successful trial. Readings:
(1) 100 Hz keypoints with the lever averaged over each video frame, or the 30 Hz series; (2)
likelihood < 0.2 dropped or kept; (3) trial frames from the trial interval, or tone onset to onset
+ 1 s + T_pull; (4) mean and s.d. over sessions or over per-mouse means; (5) R as computed or its
absolute value. The lever tip is `rightbartip`, "The tip of the lever" in the paper's code
(bdbc-nwb-explorer data/body_video.csv); `leftbartip`, the pawrest, is shown as a contrast.

In [ ]:
rows = []
for r, t in zip(task.itertuples(), TASK):
    if "corr" not in t:
        continue
    d = {"subject": r.subject, "day": int(r.day), "n_success": t["corr"][("n_success",)]}
    d.update({"|".join(k): v for k, v in t["corr"].items() if len(k) == 4})
    rows.append(d)
CR = pd.DataFrame(rows)
print("task sessions with body keypoints:", len(CR), "; with no successful trial:", int((CR.n_success == 0).sum()))
CR = CR[CR.n_success > 0]
print("sessions analysed:", len(CR), "; mice:", CR.subject.nunique())
out = []
for p in bdbc.BODY_PARTS:
    for samp, lik in [("100Hz", "kept"), ("100Hz", "dropped"), ("30Hz", "dropped")]:
        for rd in "ab":
            x = CR[f"{p}|{samp}|{lik}|{rd}"]
            for sign in ["raw", "abs"]:
                y = x if sign == "raw" else x.abs()
                pm = y.groupby(CR.subject).mean()
                out.append(dict(keypoint=p, sampling=samp, likelihood=lik, frames=rd, sign=sign, n=int(y.count()),
                                sessions_mean=y.mean(), sessions_sd=y.std(), mice_mean=pm.mean(), mice_sd=pm.std()))
KC = pd.DataFrame(out)
print(KC.round(3).to_string())

In [ ]:
lt = KC[KC.keypoint == "rightbartip"]
print("lever tip: mean R over sessions %.3f to %.3f, over mice %.3f to %.3f across readings; all within 0.98 +/- 0.02: %s"
      % (lt.sessions_mean.min(), lt.sessions_mean.max(), lt.mice_mean.min(), lt.mice_mean.max(),
         bool((lt.sessions_mean - 0.98).abs().le(0.02).all() and (lt.mice_mean - 0.98).abs().le(0.02).all())))
x = CR["rightbartip|100Hz|dropped|a"]
print("main reading (100 Hz, dropped, trial interval): mean %.3f, s.d. %.3f, min %.3f, sessions below 0.9: %d" % (x.mean(), x.std(), x.min(), (x < 0.9).sum()))
print("per mouse mean R (main reading):", x.groupby(CR.subject).mean().round(3).to_dict())

**Result: reproduced.** The lever-tip keypoint correlates with the encoder lever at a mean R within
0.98 +/- 0.02 under every reading (the printed range), over the sessions with body video and at
least one successful trial; per mouse the means are printed.

In [ ]:
rp, lp = KC[KC.keypoint == "rightpawcenter"].reset_index(drop=True), KC[KC.keypoint == "leftpawcenter"].reset_index(drop=True)
J = rp[["sampling", "likelihood", "frames", "sign"]].copy()
for lvl in ["sessions", "mice"]:
    J[f"right_{lvl}"] = rp[f"{lvl}_mean"]; J[f"left_{lvl}"] = lp[f"{lvl}_mean"]
    J[f"match_{lvl}"] = (rp[f"{lvl}_mean"] - 0.49).abs().le(0.23) & (lp[f"{lvl}_mean"] - 0.05).abs().le(0.19) & (rp[f"{lvl}_mean"] > lp[f"{lvl}_mean"])
print(J.round(3).to_string())
n_body = int(sum(t["has_body_kp"] for t in TASK))
print("sessions analysed: %d (expected 354, tolerance 0); task sessions with body keypoints: %d (the paper: 355)" % (len(CR), n_body))
x, y = CR["rightpawcenter|100Hz|dropped|a"], CR["leftpawcenter|100Hz|dropped|a"]
print("main reading: right %.3f +/- %.3f, left %.3f +/- %.3f" % (x.mean(), x.std(), y.mean(), y.std()))
print("per mouse, right minus left mean R: min %.3f, mice with right > left: %d of %d"
      % ((x.groupby(CR.subject).mean() - y.groupby(CR.subject).mean()).min(), int((x.groupby(CR.subject).mean() > y.groupby(CR.subject).mean()).sum()), CR.subject.nunique()))
pt = KC[(KC.keypoint == "leftbartip") & (KC.sign == "raw")]
print("contrast, pawrest tip (leftbartip): mean R %.3f to %.3f" % (pt.sessions_mean.min(), pt.sessions_mean.max()))

**Result: partially reproduced.** The right forepaw correlates with the lever at a mean R within
0.49 +/- 0.23 and the left forepaw at a mean R within 0.05 +/- 0.19 under every reading with R as
computed (with absolute values the left forepaw is still within tolerance), and the right exceeds
the left in every mouse. The number of sessions, an exact expected value, is 356 here against the
paper's 354: the dataset has 357 task sessions with body keypoints, one without a successful trial.
The pawrest keypoint correlates with the lever at R near 0.4, though the pawrest is described as
immobile (the printed contrast).

## fig6b claims: pull-aligned movements and cortical activity

The four claims below are stated in the paper for one unidentified example session (Figure 6). The
criterion's readings are (a) the mean over all task sessions with videos of each session's trial
average, (b) the fraction of sessions in which the statement holds (holding when above one half);
lever-pull onset is the trial table's `pull_onset` (the online 1 mm threshold, 14 ms after the
lever passes 1 mm, as the data check shows), for (a) successful trials, (b) all valid pulls
(success and failure); traces from -1 to 3 s. When no reading holds, the claim is blocked as
methods_underspecified.

In [ ]:
T6 = TASK[0]["aligned"]["t"]
PARTS = {"right forepaw": "body_rightpawcenter", "left forepaw": "body_leftpawcenter", "nose": "face_nosetip", "jaw": "face_lowerjaw"}
def sessions_with(sel, need):
    return [(r.subject, t["aligned"][sel]) for r, t in zip(task.itertuples(), TASK)
            if t["has_body_kp"] and t["aligned"][sel]["n"] >= 2 and all(k in t["aligned"][sel] for k in need)]
need = ["kpxy_" + v for v in PARTS.values()] + ["kp_" + v for v in PARTS.values()] + ["pupil_mean"]
for sel in ["success", "valid"]:
    print(sel, ": sessions with body and face keypoints and pupil and >= 2 pulls:", len(sessions_with(sel, need)),
          "; trials per session median", int(np.median([a["n"] for _, a in sessions_with(sel, need)])))

### fig6b-movement-order

> In the example session, the forepaws and the nose began moving before lever-pull onset, and the
> jaw and pupil diameter changed after it.

Criterion: each keypoint's displacement from its mean position over -1 to -0.5 s; a trace's onset
is the first time its trial average leaves the baseline mean +/- 2 s.d. (also 3 s.d.). The
criterion's "displacement ... trial average" is run both ways: the trial average of each trial's
displacement, and the displacement of the trial-averaged position. Pupil: the trial-averaged
diameter. Reproduced when both forepaws and the nose have onsets before 0 and the jaw and pupil
after 0.

In [ ]:
def onsets(a, nsd, mode):
    o = {}
    for nm, v in PARTS.items():
        tr_ = a["kp_" + v] if mode == "mean of displacements" else bdbc.displacement_of_mean(a["kpxy_" + v], T6)
        o[nm] = bdbc.onset_time(tr_, T6, nsd=nsd)
    o["pupil"] = bdbc.onset_time(a["pupil_mean"], T6, nsd=nsd)
    return o
def order_holds(o):
    return o["right forepaw"] < 0 and o["left forepaw"] < 0 and o["nose"] < 0 and o["jaw"] > 0 and o["pupil"] > 0
rows = []
for sel in ["success", "valid"]:
    al = sessions_with(sel, need)
    for mode in ["mean of displacements", "displacement of mean"]:
        for nsd in [2, 3]:
            grand = {"kp_" + v: np.nanmean([a["kp_" + v] for _, a in al], 0) for v in PARTS.values()}
            grand.update({"kpxy_" + v: np.nanmean([a["kpxy_" + v] for _, a in al], 0) for v in PARTS.values()})
            grand["pupil_mean"] = np.nanmean([a["pupil_mean"] for _, a in al], 0)
            og = onsets(grand, nsd, mode)
            per = pd.DataFrame([onsets(a, nsd, mode) for _, a in al])
            holds = per.apply(order_holds, axis=1)
            rows.append(dict(pulls=sel, trace=mode, sd=nsd, **{"grand " + k: round(v, 3) for k, v in og.items()},
                             grand_holds=order_holds(og), frac_sessions_hold=round(holds.mean(), 3),
                             jaw_before_0=round((per.jaw < 0).mean(), 3), pupil_after_0=round((per.pupil > 0).mean(), 3)))
MO = pd.DataFrame(rows)
print(MO.to_string())
al = sessions_with("success", need)
g = {nm: bdbc.displacement_of_mean(np.nanmean([a["kpxy_" + v] for _, a in al], 0), T6) for nm, v in PARTS.items()}
print("grand displacement of the mean position (pixels) at -1, -0.5, -0.2, 0, 0.5, 1 s:")
for nm, x in g.items():
    print("  %-14s" % nm, np.round(x[np.searchsorted(T6, [-1, -0.5, -0.2, -1e-9, 0.5, 1])], 2))
pg = np.nanmean([a["pupil_mean"] for _, a in al], 0)
print("  pupil diameter", np.round(pg[np.searchsorted(T6, [-1, -0.5, -0.2, -1e-9, 0.5, 1, 2])], 2), "(also at 2 s)")

**Result: blocked (methods_underspecified).** Under no reading does the order hold: in the
mean over sessions every keypoint, the jaw included, leaves its baseline band before lever-pull
onset, and the fraction of sessions in which the full order holds is far below one half (the
table). The jaw displacement grows before 0 and peaks after it, and the pupil dilates after onset
(the printed traces). As the paper states the order for one session that cannot be identified, the
claims file blocks the claim when no reading holds.

### fig6b-motor-somatosensory-after-reward

> In the example session, MOs activity returned quickly to baseline after the pull, while MOp and
> the upper-limb and nose somatosensory areas (SSp-ul, SSp-n) stayed slightly above baseline after
> reward delivery.

Criterion: trial-averaged corrected dF/F aligned to lever-pull onset, baseline -1 to 0 s; after
reward read as (a) 1 to 3 s after onset, (b) 0.5 to 2 s after reward delivery in successful trials
(reward from the 5 kHz reward TTL, the first within 2 s of the pull onset); hemisphere left, right
or mean. Reproduced when MOs's post-reward value minus baseline is below each of MOp's, SSp-ul's
and SSp-n's, and those three are above baseline.

In [ ]:
def roi_val(X, area, hemi):
    if hemi == "mean":
        return (X[..., ROI.index(area + "_l")] + X[..., ROI.index(area + "_r")]) / 2
    return X[..., ROI.index(area + "_" + hemi)]
AREAS = ["MOs", "MOp", "SSp_ul", "SSp_n"]
def ms_holds(v):
    return all(v["MOs"] < v[a] for a in AREAS[1:]) and all(v[a] > 0 for a in AREAS[1:])
rows = []
for sel in ["success", "valid"]:
    al = [a for _, a in sessions_with(sel, ["dff_mean"])]
    D = np.array([a["dff_mean"] for a in al]); base = np.nanmean(D[:, T6 < 0, :], 1)
    posts = {"(a) 1-3 s after onset": np.nanmean(D[:, (T6 >= 1) & (T6 < 3), :], 1) - base}
    if sel == "success":
        posts["(b) 0.5-2 s after reward"] = np.array([a.get("post_reward_minus_base", np.full(44, np.nan)) for a in al])
    for pn, X in posts.items():
        for hemi in ["l", "r", "mean"]:
            v = {a: roi_val(X, a, hemi) for a in AREAS}
            g = {a: np.nanmean(x) for a, x in v.items()}
            ok = [ms_holds({a: v[a][i] for a in AREAS}) for i in range(len(X)) if np.isfinite(v["MOs"][i])]
            rows.append(dict(pulls=sel, window=pn, hemisphere=hemi, n=len(ok), **{a: round(g[a], 4) for a in AREAS},
                             grand_holds=ms_holds(g), frac_sessions_hold=round(np.mean(ok), 3)))
MS = pd.DataFrame(rows)
print(MS.to_string())

**Result: reproduced.** In the mean over sessions MOs is closer to its baseline than MOp, SSp-ul
and SSp-n, and those three stay above baseline, under every window, pull set and hemisphere (the
table), which is reading (a). Session by session, reading (b), the full pattern holds in 38% to 60%
of sessions: in more than half only for the reward-aligned window.

### fig6b-sensory-areas-track-pupil

> In the example session, OB, RSPagl, AUDs and VISp had activity time courses that closely matched
> pupil dilation.

Criterion: over -1 to 3 s around lever-pull onset; reproduced when each of the four areas' trial
averages has a higher Pearson correlation with the trial-averaged pupil diameter than with the
trial-averaged lever position, and MOp's does not; hemisphere left, right or mean.

In [ ]:
FOUR = ["OB", "RSPagl", "AUDs", "VISp"]
rows = []
for sel in ["success", "valid"]:
    al = [a for _, a in sessions_with(sel, ["dff_mean", "pupil_mean"])]
    D = np.array([a["dff_mean"] for a in al]); pup = np.array([a["pupil_mean"] for a in al]); lev = np.array([a["lever_mean"] for a in al])
    for hemi in ["l", "r", "mean"]:
        g = {}
        for area in FOUR + ["MOp"]:
            x = np.nanmean(roi_val(D, area, hemi), 0)
            g[area] = (bdbc.nan_pearson(x, np.nanmean(pup, 0)), bdbc.nan_pearson(x, np.nanmean(lev, 0)))
        grand = all(g[a][0] > g[a][1] for a in FOUR) and not g["MOp"][0] > g["MOp"][1]
        per = []
        for i in range(len(al)):
            c = {a: (bdbc.nan_pearson(roi_val(D[i], a, hemi), pup[i]), bdbc.nan_pearson(roi_val(D[i], a, hemi), lev[i])) for a in FOUR + ["MOp"]}
            per.append(all(c[a][0] > c[a][1] for a in FOUR) and not c["MOp"][0] > c["MOp"][1])
        rows.append(dict(pulls=sel, hemisphere=hemi, n=len(al), **{f"{a} r(pupil)/r(lever)": "%.2f/%.2f" % g[a] for a in FOUR + ["MOp"]},
                         grand_holds=grand, frac_sessions_hold=round(np.mean(per), 3)))
PU = pd.DataFrame(rows)
print(PU.to_string())

**Result: blocked (methods_underspecified).** In the mean over sessions OB follows the pupil
(r about 0.95 with the pupil against about 0.63 with the lever), but RSPagl, AUDs and VISp follow the
lever more closely than the pupil, in each hemisphere and for both pull sets (the table), and the
full pattern holds in at most 1.4% of sessions, so no reading holds; as the paper states it for an
unidentified example session, the criterion blocks the claim.

### fig6b-variability-decrease

> Cortical activity varied less across trials after lever-pull onset than before it.

Criterion: across-trial s.d. of corrected dF/F at each time point, averaged over a window, per
ROI; before -1 to 0 s, after (a) 0 to 1 s, (b) 0 to 3 s; reproduced when the after value is lower
in more than half of the 44 ROIs, with a Wilcoxon signed-rank test over ROIs reported.

In [ ]:
rows = []
for sel in ["success", "valid"]:
    al = [a for _, a in sessions_with(sel, ["dff_sd"])]
    SD = np.array([a["dff_sd"] for a in al]); pre = np.nanmean(SD[:, T6 < 0, :], 1)
    for nm, (a0, a1) in [("(a) 0-1 s", (0, 1)), ("(b) 0-3 s", (0, 3))]:
        post = np.nanmean(SD[:, (T6 >= a0) & (T6 < a1), :], 1)
        gpre, gpost = np.nanmean(pre, 0), np.nanmean(post, 0)
        rows.append(dict(pulls=sel, after=nm, n_sessions=len(al), rois_lower=int((gpost < gpre).sum()), wilcoxon_p=wilcoxon(gpost, gpre).pvalue,
                         median_ratio_after_before=np.median(gpost / gpre), frac_sessions_majority_lower=np.mean((post < pre).sum(1) > 22)))
VD = pd.DataFrame(rows)
print(VD.to_string())
sub_frac = pd.Series([np.mean((np.nanmean(a["dff_sd"][(T6 >= 0) & (T6 < 1)], 0) < np.nanmean(a["dff_sd"][T6 < 0], 0))) > 0.5
                      for _, a in sessions_with("success", ["dff_sd"])], index=[s for s, _ in sessions_with("success", ["dff_sd"])])
print("per mouse, fraction of sessions with most ROIs lower (successful pulls, 0-1 s): min %.2f, median %.2f" % (sub_frac.groupby(level=0).mean().min(), sub_frac.groupby(level=0).mean().median()))

**Result: reproduced.** In the mean over sessions the across-trial s.d. after lever-pull onset is
below the s.d. before it in all or nearly all of the 44 ROIs, under both windows and pull sets
(Wilcoxon p far below 0.05), and the same holds in most sessions (the table).

## Summary

| Claim | Outcome |
|---|---|
| cohort-sex-age | reproduced |
| task-sessions-collected | reproduced |
| imaging-frame-counts | reproduced |
| training-breaks | reproduced |
| fig3c-response-rate-increase | reproduced |
| fig3c-success-rate-constant | blocked: methods underspecified (the two readings of "constant" disagree) |
| fig3d-tpull-clusters | reproduced |
| fig3e-cluster-a-high-tpull | reproduced |
| fig3e-cluster-b-similar-rates | reproduced (under one set of readings) |
| methods-no-signal-degradation | reproduced |
| fig4ab-landmark-validation | blocked: data missing |
| fig4d-visual-map | reproduced |
| fig4d-tactile-map | reproduced |
| fig4d-auditory-map | reproduced |
| figs2b-no-aberrant-transients | reproduced |
| fig5abc-dlc-error | blocked: data missing |
| fig5f-lever-tip-correlation | reproduced |
| fig5f-forepaw-correlation | partially reproduced (356 sessions against 354) |
| fig6b-movement-order | blocked: methods underspecified |
| fig6b-motor-somatosensory-after-reward | reproduced |
| fig6b-sensory-areas-track-pupil | blocked: methods underspecified |
| fig6b-variability-decrease | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The raw blue fluorescence is sampled from 8 frames
per session; the sensory maps use the ROI series only (the pixel maps do not fit the time limit);
the Figure 6 claims are tested over all sessions, as the example session is not identified;
the stimulus identity of the sensory-mapping trials is inferred from the stimulus pattern and the
evoked maps, as the files do not name the codes.

What was found about the dataset. The excitation wavelengths of the two imaging planes are
swapped in every file; the lever's unit is "a.u." where the code says mm; the sensory-mapping trial
codes have no description, six sessions have no `stim_visual` channel and carry the visual train on
`stim_whisker` and the tactile stimulus on `stim_auditory`, with no third stimulus, and in the other
sessions the trial order is visual, auditory, tactile against the Methods' visual, tactile,
auditory; reward TTLs outnumber success labels in more than half of the task sessions; some pupil diameters are infinite or of 1e5 pixels and more; four sensory-mapping sessions
have 36000 frames, not 27000; 357 task sessions have body keypoints against the paper's 355; the
session-initial T_pull does not follow the Methods' rule (previous final minus 100 ms) in about a
third of consecutive sessions.

In [ ]:
ok = B.sort_values(["subject", "day"]).copy()
ok["prev_final"] = ok.groupby("subject").tpull_final.shift(); ok["prev_day"] = ok.groupby("subject").day.shift()
ok = ok[ok.prev_day == ok.day - 1]
rule = np.isclose(ok.tpull_first, np.maximum(ok.prev_final - 0.1, 0), atol=0.002)
print("initial T_pull = previous final - 100 ms (floored at 0) in %d of %d consecutive session pairs" % (rule.sum(), len(ok)))
print(pd.crosstab(ok.prev_final.round(2), ok.tpull_first.round(2)).to_string())